# Day 3: Batch Ingestion (X12 EDI Claims & Debezium CDC)

This notebook simulates parsing synthetic raw EDI lines and Debezium JSON records into structured Bronze output.

In [ ]:
!pip install pyspark pyiceberg
import os
import tempfile

# Set Spark version if needed
os.environ['SPARK_VERSION'] = '3.5'

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp, from_json
from pyspark.sql.types import StructType, StructField, StringType, MapType

# Initialize Spark Session with Iceberg configurations
spark = SparkSession.builder \
    .appName("Day03_Batch_CDC_Simulation") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", "/tmp/iceberg_warehouse") \
    .getOrCreate()

print("Spark Session initialized.")

In [ ]:
# Create Bronze Database/Table if not exists
spark.sql("CREATE DATABASE IF NOT EXISTS local.ehdip_bronze_db")

spark.sql("""
CREATE TABLE IF NOT EXISTS local.ehdip_bronze_db.raw_payloads (
    payload_id STRING,
    source_system_id STRING,
    raw_payload_json STRING,
    ingestion_timestamp TIMESTAMP
)
USING iceberg
PARTITIONED BY (days(ingestion_timestamp))
""")

# --- 1. Simulate X12 Batch Ingestion ---
x12_data = [("ISA*00*          *00*          *ZZ*1234567890     *ZZ*0987654321     *100101*1200*U*00401*000000001*0*T*:",)]
x12_df = spark.createDataFrame(x12_data, ["value"])

processed_x12 = x12_df \
    .withColumn("payload_id", lit("x12-batch-id-1")) \
    .withColumn("source_system_id", lit("EDI_X12_BATCH")) \
    .withColumnRenamed("value", "raw_payload_json") \
    .withColumn("ingestion_timestamp", current_timestamp())

processed_x12.write \
    .format("iceberg") \
    .mode("append") \
    .save("local.ehdip_bronze_db.raw_payloads")

print("X12 Data Appended to Bronze.")

# --- 2. Simulate Debezium CDC Ingestion ---
cdc_data = [
    ('{"payload": {"after": {"id": "pat-123", "name": "John Doe"}, "source": {"db": "postgres"}, "op": "c"}}',),
    ('{"payload": {"after": {"id": "pat-456", "name": "Jane Doe"}, "source": {"db": "postgres"}, "op": "u"}}',)
]
cdc_df = spark.createDataFrame(cdc_data, ["json_str"])

debezium_schema = StructType([
    StructField("payload", StructType([
        StructField("after", MapType(StringType(), StringType()), True),
        StructField("source", MapType(StringType(), StringType()), True),
        StructField("op", StringType(), True)
    ]), True)
])

parsed_cdc_df = cdc_df \
    .withColumn("data", from_json(col("json_str"), debezium_schema)) \
    .filter(col("data.payload.op").isin('c', 'u', 'd')) \
    .select(
        col("data.payload.after.id").alias("payload_id"),
        col("data.payload.source.db").alias("source_system_id"),
        col("json_str").alias("raw_payload_json"),
        current_timestamp().alias("ingestion_timestamp")
    )

parsed_cdc_df.write \
    .format("iceberg") \
    .mode("append") \
    .save("local.ehdip_bronze_db.raw_payloads")

print("CDC Data Appended to Bronze.")

# Verify Bronze Table contents
spark.sql("SELECT * FROM local.ehdip_bronze_db.raw_payloads").show(truncate=False)